In [1]:
import pandas as pd
import numpy as np

In [2]:
import inflect
from functools import lru_cache

inflect_engine = inflect.engine()

@lru_cache(maxsize=50_000)
def singularize_word(word: str) -> str:
    normalized = word.strip().lower()
    return inflect_engine.singular_noun(normalized) or normalized
singularize_word("antidepressants")   # antidepressant


'antidepressant'

#### NOTE
For translation on drug level we work with all abstracts regardless if full-text was found!

In [3]:
# from /04_normalization/Analyse_Combine_Linked_Entities.ipynb
FILE_PRECLINICAL_LINKING = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin.csv"
FILE_CLINICAL_LINKING = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_clin.csv"
FILE_CLINICAL_METADATA = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/06_preclin_clinic_join/data/clinical/clinical_nct_docs_metadata_20240313.csv"

OUTPUT_DIR = "06_preclin_clinic_join/data/joined_data/"

conditions_col_to_use = "merged_mondo_label" # #"linkbert_mapped_conditions" # disease_term_mondo_norm
drugs_col_to_use =  "merged_umls_label" #"linkbert_mapped_drugs" #"drug_term_umls_norm"

conditions_col_to_use_clinical =  "merged_mondo_label"#"linkbert_aact_mapped_conditions" #"disease_term_mondo_norm"
drugs_col_to_use_clinical =  "merged_umls_label" #"linkbert_aact_mapped_drugs" # "drug_term_umls_norm"


In [4]:
def count_unique_from_pipe_column(df, column):
    """
    Count unique items and their frequencies in a DataFrame column containing '|' separated values.

    Returns:
        total_unique (int): total number of unique non-empty terms
        freq_df (pd.DataFrame): columns ['term', 'n_articles']
                               where 'n_articles' = number of unique PMIDs (rows) mentioning that term
    """
    import pandas as pd

    # explode values
    all_items = (
        df[[column, "PMID"]]
        .dropna(subset=[column])
        .assign(**{column: df[column].astype(str).str.split("|")})
        .explode(column)
    )
    all_items[column] = all_items[column].str.strip()
    all_items = all_items[all_items[column] != ""]

    # count how many distinct PMIDs mention each term
    freq = (
        all_items.groupby(column)["PMID"]
        .nunique()
        .reset_index(name="n_articles")
        .sort_values("n_articles", ascending=False)
    )

    total_unique = freq.shape[0]
    return total_unique, freq


## Preclinical Data

In [5]:
# --- Load Preclinical Data ---
preclinical_df = pd.read_csv(FILE_PRECLINICAL_LINKING)
print(f"Shape of preclinical_df: {preclinical_df.shape}, {preclinical_df.PMID.nunique()} unique PMIDs")

preclinical_df.head()

Shape of preclinical_df: (540999, 14), 540999 unique PMIDs


,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
0,31733831,asthma,asthma,MONDO:0004979,asthma,-1,asthma,MONDO:0004979,isorhynchophylline,isorhynchophylline,C0245133,-1,isorhynchophylline,C0245133
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1
2,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,-1,systemic lupus erythematosus,MONDO:0007915,hla-g2|g2,HLA-G2 Isoform|g2,C0967254|-1,-1,HLA-G2 Isoform|g2,C0967254|-1
3,31733940,cognitive impairment,cognitive disorder,MONDO:0002039,cognitive disorder,-1,cognitive disorder,MONDO:0002039,minocycline,Minocycline,C0026187,-1,Minocycline,C0026187
4,31734027,cumulative peripheral neuropathy|oxaliplatin-i...,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620,cumulative peripheral neuropathy|oxaliplatin-i...,-1,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620,tadalafil|phosphodiesterase type 5 inhibitor t...,Tadalafil|Tadalafil,C1176316|C1176316,-1,Tadalafil,C1176316


In [6]:
preclinical_df[preclinical_df["merged_umls_label"].str.contains("cariprazine hydrochloride", case=False, na=False)].head()

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
279166,39364867,schizophrenia,schizophrenia,MONDO:0005090,schizophrenia,-1,schizophrenia,MONDO:0005090,cariprazine|cariprazine -|cariprazine hydrochl...,Cariprazine|Cariprazine|cariprazine hydrochloride,C2936870|C2936870|C4058574,Cariprazine,Cariprazine|cariprazine hydrochloride,C2936870|C4058574


In [7]:
n_unique, freq = count_unique_from_pipe_column(preclinical_df, "merged_umls_label")
print(f"Unique count: {n_unique}")

Unique count: 292514


In [8]:
freq[freq["merged_umls_label"].str.contains("cariprazine hydrochloride", case=False, na=False)].head()

,merged_umls_label,n_articles
101158,cariprazine hydrochloride,1


In [9]:
freq.head()

,merged_umls_label,n_articles
35534,Dexamethasone,5806
29728,Acetylcysteine,4559
46202,NG-Nitroarginine Methyl Ester,4306
52975,Sirolimus,4107
35945,Doxorubicin,4021


In [10]:
save_path = f"out/unique_drug_terms_{n_unique}.csv" # used for the FDA filtering of drugs to further process
freq.to_csv(save_path, index=False)

#### flatten drug/disease

In [11]:
# Split and explode conditions and drugs
preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.split("|")
preclinical_df = preclinical_df.explode(conditions_col_to_use, ignore_index=True)

cols_to_explode = [
    drugs_col_to_use,     # e.g. drug names
    "merged_umls_termid",             # IDs
]

for col in cols_to_explode:
    preclinical_df[col] = preclinical_df[col].astype(str).str.split("|")

preclinical_df = preclinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique drugs before length filter: {preclinical_df[drugs_col_to_use].nunique()}")

preclinical_df = preclinical_df[preclinical_df[drugs_col_to_use].fillna("").str.len() > 2]

print(f"Unique drugs: {preclinical_df[drugs_col_to_use].nunique()}")
preclinical_df = preclinical_df.drop_duplicates()

# Strip whitespace and convert to lowercase
preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.strip().str.lower()
preclinical_df[drugs_col_to_use] = preclinical_df[drugs_col_to_use].str.strip().str.lower()

Unique drugs before length filter: 292514
Unique drugs: 291465


In [12]:
preclinical_df.head()

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
0,31733831,asthma,asthma,MONDO:0004979,asthma,-1,asthma,MONDO:0004979,isorhynchophylline,isorhynchophylline,C0245133,-1,isorhynchophylline,C0245133
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,"mirn1192 microrna, mouse",C3849422
2,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,antgomir-1192,-1
3,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,agomir-1192,-1
4,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,-1,systemic lupus erythematosus,MONDO:0007915,hla-g2|g2,HLA-G2 Isoform|g2,C0967254|-1,-1,hla-g2 isoform,C0967254


In [13]:
mapping_lookup = dict(
    zip(
        preclinical_df["merged_umls_label"],
        preclinical_df["merged_umls_termid"]
    )
)


In [14]:
preclinical_drugs = set(preclinical_df['merged_umls_label'])

In [15]:
len(preclinical_drugs)

291457

## Clinical Data

In [16]:
# --- Load Clinical Data ---
clinical_df = pd.read_csv(FILE_CLINICAL_LINKING)
print(f"Shape of clinical_df: {clinical_df.shape}, {clinical_df.nct_id.nunique()} unique NCTIDs")

clinical_df[conditions_col_to_use_clinical] = clinical_df[conditions_col_to_use_clinical].str.split("|")
clinical_df = clinical_df.explode(conditions_col_to_use_clinical, ignore_index=True)

cols_to_explode = [
    drugs_col_to_use_clinical,     # e.g. drug names
    "merged_umls_termid",             # IDs
]

for col in cols_to_explode:
    clinical_df[col] = clinical_df[col].astype(str).str.split("|")

clinical_df = clinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique drugs before length filter: {clinical_df[drugs_col_to_use_clinical].nunique()}")
clinical_df = clinical_df[clinical_df[drugs_col_to_use_clinical].fillna("").str.len() > 2]
print(f"Unique drugs: {clinical_df[drugs_col_to_use_clinical].nunique()}")

# Strip whitespace and convert to lowercase
clinical_df[conditions_col_to_use_clinical] = clinical_df[conditions_col_to_use_clinical].str.strip().str.lower()
clinical_df[drugs_col_to_use_clinical] = clinical_df[drugs_col_to_use_clinical].str.strip().str.lower()

# Create disease-drug key
clinical_df['disease<>drug'] = (
    clinical_df[conditions_col_to_use_clinical] + " <> " + clinical_df[drugs_col_to_use_clinical]
)


# Load and merge clinical metadata (phase + status)
metadata_df = pd.read_csv(FILE_CLINICAL_METADATA)[['nct_id', 'phase', 'overall_status']]
metadata_df = metadata_df.drop_duplicates()

clinical_df = clinical_df.merge(metadata_df, on='nct_id', how='left')

Shape of clinical_df: (18609, 14), 18609 unique NCTIDs
Unique drugs before length filter: 12904
Unique drugs: 12705


In [17]:
clinical_df.shape

(124290, 17)

In [18]:
clinical_df.head()

,nct_id,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,disease<>drug,phase,overall_status
0,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicidal,-1|-1|-1,ketamine,Ketamine,C0022614,-1,ketamine,C0022614,suicidal <> ketamine,Phase 2,Withdrawn
1,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicidal ideation,-1|-1|-1,ketamine,Ketamine,C0022614,-1,ketamine,C0022614,suicidal ideation <> ketamine,Phase 2,Withdrawn
2,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicide,-1|-1|-1,ketamine,Ketamine,C0022614,-1,ketamine,C0022614,suicide <> ketamine,Phase 2,Withdrawn
3,NCT05216770,laryngeal dystonia|voice tremor|Tremor,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644,spasmodic dystonia|voice tremor|obsolete rare ...,focal dystonia|-1,spasmodic dystonia,MONDO:0000485|-1|MONDO:0017644|MONDO:0000477,Laryngeal sensory block with topical bupivacaine,Laryngeal sensory block with topical bupivacaine,-1,-1,laryngeal sensory block with topical bupivacaine,-1,spasmodic dystonia <> laryngeal sensory block ...,Early Phase 1,Recruiting
4,NCT05216770,laryngeal dystonia|voice tremor|Tremor,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644,spasmodic dystonia|voice tremor|obsolete rare ...,focal dystonia|-1,voice tremor,MONDO:0000485|-1|MONDO:0017644|MONDO:0000477,Laryngeal sensory block with topical bupivacaine,Laryngeal sensory block with topical bupivacaine,-1,-1,laryngeal sensory block with topical bupivacaine,-1,voice tremor <> laryngeal sensory block with t...,Early Phase 1,Recruiting


In [19]:
clinical_drugs = set(clinical_df['merged_umls_label'])
len(clinical_drugs)

12520

## Aggregate for merging

In [20]:
clinical_doc_id_col = "nct_id"
clinical_key_col = drugs_col_to_use_clinical

In [21]:
clinical_unique = (
        clinical_df
        .sort_values(clinical_doc_id_col)  # optional: enforce a deterministic order
        .drop_duplicates(subset=[clinical_key_col, clinical_doc_id_col], keep='first')
    )
# === Aggregate clinical data by key ===
clinical_agg = (
        clinical_unique
        .groupby(clinical_key_col)
        .agg({
            clinical_doc_id_col: list,
            'phase':              list,
            'overall_status':     list
        })
        .reset_index()
        )
    

# Rename columns for clarity and consistency
clinical_agg.rename(columns={
    clinical_key_col: 'normalized_key',
    clinical_doc_id_col: 'clinical_doc_ids'
}, inplace=True)

# Add count of clinical documents per key
clinical_agg['clinical_count'] = clinical_agg['clinical_doc_ids'].apply(len)

In [22]:
clinical_agg

,normalized_key,clinical_doc_ids,phase,overall_status,clinical_count
0,""" funny "" channel blocker",[NCT01761825],[Phase 2],[Unknown status],1
1,"""ketamine"" and ""dexmedetomidine"" and ""propofol...",[NCT04018157],[Early Phase 1],[Completed],1
2,"""nycoplus magnesium"" (120 mg x 3 daily for 2 w...",[NCT00525317],[Phase 4],[Completed],1
3,"""raltegravir"" and ""zidovudine""",[NCT02655471],[Early Phase 1],[Completed],1
4,"""raylis""",[NCT01866995],[Phase 3],[Unknown status],1
...,...,...,...,...,...
12515,δ9 -,[NCT02051387],[Phase 1],[Completed],1
12516,δ9-tetrahydracannabinol,[NCT03066193],[Phase 2],[Completed],1
12517,μ-opioid agonist,[NCT03958396],[Early Phase 1],[Completed],1
12518,∆-9thc,[NCT03639064],[Phase 2],[Unknown status],1


In [23]:
preclinical_key_col = drugs_col_to_use
preclinical_doc_id_col = "PMID"
preclinical_disease_col = conditions_col_to_use

In [24]:
# === Aggregate preclinical data by key ===
preclinical_agg = preclinical_df.groupby(preclinical_key_col).agg({
    preclinical_doc_id_col: list,
    preclinical_disease_col: list, #'first',
    #preclinical_drug_col: 'first'
}).reset_index()

# Rename columns for consistency
preclinical_agg.rename(columns={
    preclinical_key_col: 'normalized_key',
    preclinical_doc_id_col: 'preclinical_doc_ids',
    preclinical_disease_col: 'disease',
    #preclinical_drug_col: 'drug'
}, inplace=True)

# Add count of preclinical documents per key
preclinical_agg['preclinical_count'] = preclinical_agg['preclinical_doc_ids'].apply(len)

In [25]:
preclinical_agg.head()

,normalized_key,preclinical_doc_ids,disease,preclinical_count
0,""" 528","[13413150, 13413150]","[trypanosomiasis, bovine, trypanosomiasis, non...",2
1,""" ib4",[9390653],[traumatic brain injury],1
2,""" k + channel opener","[2345562, 2345562, 2345562]","[hyperkalemic periodic paralysis, hypokalemic ...",3
3,""" metformin-like "" amp-activated protein kinas...","[34975743, 34975743]","[hypoglycemia, diabetes mellitus]",2
4,""" nonglycosaminoglycan "" thrombolytic agent",[2766306],[superficial urinary bladder carcinoma],1


## Merge clinical and preclinical on drug

In [26]:
merged_df = pd.merge(clinical_agg, preclinical_agg, on='normalized_key', how='outer')

In [27]:
def clean_and_sort_study_data(df):
    """
    Cleans and sorts a merged clinical/preclinical DataFrame by removing invalid rows
    and ordering by study counts.

    Specifically:
    - Sorts the DataFrame in descending order of 'clinical_count' and 'preclinical_count'.
    - Removes rows where either 'clinical_count' or 'preclinical_count' is NaN.
    - Removes rows where the 'drug' name is missing or has 3 or fewer characters.
    - Prints the shape before and after filtering for transparency.

    Parameters:
    ----------
    df : pd.DataFrame
        A DataFrame containing 'clinical_count', 'preclinical_count', and 'drug'.

    Returns:
    -------
    pd.DataFrame
        The cleaned and sorted DataFrame.
    """
    print(f'Input shape of merged for cleaning: {df.shape}')

    # Sort by descending clinical and preclinical study counts
    sorted_df = df.sort_values(
        by=['clinical_count', 'preclinical_count'],
        ascending=[False, False]
    )

    # Remove rows with missing study counts
    filtered_df = sorted_df.dropna(subset=['clinical_count', 'preclinical_count'])
    print(f'Shape after filtering NaN counts: {filtered_df.shape}')

    # Remove rows with invalid or too-short drug names
    #filtered_df = filtered_df[
    #    filtered_df['drug'].apply(lambda x: isinstance(x, str) and len(x.strip()) > 3)
    #]
    #print(f'Shape after filtering short or invalid drugs: {filtered_df.shape}')

    print(f'Shape after filtering empty counts and short drugs: {filtered_df.shape}')
    return filtered_df

In [28]:
phase_order = {
    'Early Phase 1': 0,
    'Phase 1': 1,
    'Phase 1/Phase 2': 1.5,
    'Phase 2': 2,
    'Phase 2/Phase 3': 2.5,
    'Phase 3': 3,
    'Phase 4': 4,
    'Not Applicable': -1  # Lowest value to ignore in max comparison
}

# Get max phase for each row
def get_max_phase(phases):
    max_val = -1
    max_phase = 'Not Applicable'
    for phase in phases:
        val = phase_order.get(phase, -1)
        if val > max_val:
            max_val = val
            max_phase = phase
    return max_phase


In [29]:
merged_df.head()

,normalized_key,clinical_doc_ids,phase,overall_status,clinical_count,preclinical_doc_ids,disease,preclinical_count
0,""" funny "" channel blocker",[NCT01761825],[Phase 2],[Unknown status],1.0,NaN,NaN,NaN
1,"""ketamine"" and ""dexmedetomidine"" and ""propofol...",[NCT04018157],[Early Phase 1],[Completed],1.0,NaN,NaN,NaN
2,"""nycoplus magnesium"" (120 mg x 3 daily for 2 w...",[NCT00525317],[Phase 4],[Completed],1.0,NaN,NaN,NaN
3,"""raltegravir"" and ""zidovudine""",[NCT02655471],[Early Phase 1],[Completed],1.0,NaN,NaN,NaN
4,"""raylis""",[NCT01866995],[Phase 3],[Unknown status],1.0,NaN,NaN,NaN


In [30]:
filtered_df = clean_and_sort_study_data(merged_df)
filtered_df.head()

Input shape of merged for cleaning: (299192, 8)
Shape after filtering NaN counts: (4785, 8)
Shape after filtering empty counts and short drugs: (4785, 8)


,normalized_key,clinical_doc_ids,phase,overall_status,clinical_count,preclinical_doc_ids,disease,preclinical_count
9128,placebo,"[NCT00006289, NCT00030147, NCT00035529, NCT000...","[Phase 2, Phase 4, Phase 2, Phase 2, Phase 4, ...","[Terminated, Completed, Terminated, Completed,...",330.0,"[33767693, 33767693, 33767693, 33767693]","[severe sepsis, septic shock, systemic inflamm...",4.0
10014,risperidone,"[NCT00014001, NCT00015548, NCT00018291, NCT000...","[Phase 4, Not Applicable, Not Applicable, Phas...","[Completed, Completed, Completed, Completed, C...",295.0,"[26371055, 32866524, 14638382, 24317442, 36744...","[tinnitus, schizophrenia, alzheimer disease, s...",1063.0
6639,levodopa,"[NCT00004576, NCT00006077, NCT00006337, NCT000...","[Phase 2, Phase 2, Phase 2, Phase 2, Phase 2, ...","[Completed, Completed, Completed, Completed, C...",287.0,"[31915844, 31920431, 11860157, 11860478, 11860...","[parkinson disease, parkinson disease, parkins...",3326.0
6339,ketamine,"[NCT00088699, NCT00122759, NCT00137085, NCT002...","[Phase 1/Phase 2, Not Applicable, Not Applicab...","[Completed, Unknown status, Completed, Complet...",275.0,"[31738939, 31740575, 31187076, 31187076, 31870...","[chronic pain syndrome, alcohol abuse, anhedon...",3488.0
8429,olanzapine,"[NCT00014001, NCT00015548, NCT00034580, NCT000...","[Phase 4, Not Applicable, Phase 4, Phase 4, Ph...","[Completed, Completed, Completed, Completed, C...",226.0,"[32866524, 19359144, 19359144, 19359144, 19359...","[schizophrenia, major depressive disorder, aut...",833.0


In [31]:
unique_preclinical_docs = (
    filtered_df["preclinical_doc_ids"]
    .dropna()
    .explode()
    .nunique()
)

print("Total unique preclinical doc IDs:", unique_preclinical_docs)


Total unique preclinical doc IDs: 335948


In [32]:
filtered_df['max_phase'] = filtered_df['phase'].apply(get_max_phase)

# Identify studies with Phase 3 or 4 trials
filtered_df['at_least_one_phase3'] = filtered_df['phase'].apply(
    lambda phases: any("Phase 3" in str(p) for p in phases)
)

filtered_df["at_least_one_phase3_completed"] = filtered_df.apply(
    lambda row: any(
        ("Phase 3" in str(p)) and (str(s) == "Completed")
        for p, s in zip(row["phase"], row["overall_status"])
    ),
    axis=1
)

filtered_df['at_least_one_phase4'] = filtered_df['phase'].apply(
    lambda phases: any("Phase 4" in str(p) for p in phases)
)

## Add FDA info

In [33]:
fda_drugs = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/df_ds_drugs_with_FDA_info.csv")

In [34]:
fda_drugs_agg = (
    fda_drugs
    .groupby("merged_umls_label", as_index=False)
    .agg({
        "year": lambda x: ", ".join(
            map(str, sorted({int(y) for y in x.dropna()}))
        ),
        "sponsor_name": lambda x: ", ".join(
            sorted({s.strip() for s in x.dropna() if s.strip()})
        ),
        "application_number": lambda x: ", ".join(
            sorted({str(a).strip() for a in x.dropna() if str(a).strip()})
        ),
    })
)
fda_drugs_agg = fda_drugs_agg.add_prefix("fda_")
fda_drugs_agg['fda_AP']=True
fda_drugs_agg.fda_merged_umls_label.nunique()


2745

In [35]:
fda_drugs_agg.head()

,fda_merged_umls_label,fda_year,fda_sponsor_name,fda_application_number,fda_AP
0,(+)-Tolterodine,"1998, 2000, 2012, 2013, 2015, 2016, 2017, 2019...","AJANTA PHARMA LTD, APOTEX CORP, AUROBINDO PHAR...","ANDA077006, ANDA079141, ANDA200164, ANDA201486...",True
1,(+-)-Milnacipran,"2009, 2016, 2024","ABBVIE, AMNEAL PHARMS, BRECKENRIDGE, HETERO LA...","ANDA205071, ANDA205081, ANDA205147, NDA022256",True
2,(-)-Pramipexole,"1997, 2008, 2010, 2012, 2013, 2014, 2015, 2016...","ACTAVIS ELIZABETH, ALEMBIC, AUROBINDO PHARMA L...","ANDA077724, ANDA077854, ANDA078551, ANDA078894...",True
3,(18F)FES,2020,GE HEALTHCARE,NDA212155,True
4,"1,2-Distearoyllecithin",2014,BRACCO,NDA203684,True


In [36]:
fda_drugs = set(fda_drugs_agg['fda_merged_umls_label'])

#### sanity check preclin, clin, fda

In [37]:
# pairwise overlaps
fda_drugs = {str(x).strip().lower() for x in fda_drugs if x}
clinical_drugs = {str(x).strip().lower() for x in clinical_drugs if x}
preclinical_drugs = {str(x).strip().lower() for x in preclinical_drugs if x}

fda_clinical = fda_drugs & clinical_drugs
fda_preclinical = fda_drugs & preclinical_drugs
clinical_preclinical = clinical_drugs & preclinical_drugs

# all three
all_three = fda_drugs & clinical_drugs & preclinical_drugs

print("FDA ∩ Clinical:", len(fda_clinical))
print("FDA ∩ Preclinical:", len(fda_preclinical))
print("Clinical ∩ Preclinical:", len(clinical_preclinical))
print("All three:", len(all_three))


FDA ∩ Clinical: 1307
FDA ∩ Preclinical: 2743
Clinical ∩ Preclinical: 4785
All three: 1307


In [38]:
fda_preclin_not_clin = (fda_drugs & preclinical_drugs) - clinical_drugs
len(fda_preclin_not_clin)

1436

In [39]:
clin_not_preclin = clinical_drugs - preclinical_drugs
len(clin_not_preclin)

7735

In [40]:
4785+1436

6221

In [41]:
list(clin_not_preclin)[:10]

['only aspirin medication',
 'levoleucovorin calcium',
 'xeomin injectable product',
 'testofen 300mg',
 'pascorbin',
 'aci-7104. 056',
 'chronic opioid',
 'nfl-101 dose 1',
 'f-ipv',
 'α2 adrenoreceptor agonist']

In [42]:
list(fda_preclin_not_clin)[:10]

['levobetaxolol',
 'docetaxel',
 'pertuzumab',
 'calcitonin human',
 'olaratumab',
 'mafenide acetate',
 'ensartinib',
 'iodipamide',
 'capreomycin',
 'amen']

#### add FDA to ds

In [43]:
fda_drugs_agg["fda_merged_umls_label"] = (
    fda_drugs_agg["fda_merged_umls_label"].astype(str).str.strip().str.lower()
)

filtered_df = filtered_df.merge(
    fda_drugs_agg,
    how="left",
    left_on="normalized_key",
    right_on="fda_merged_umls_label"
)
filtered_df["fda_AP"] = filtered_df["fda_AP"].fillna(False).astype(bool)


In [44]:
filtered_df[filtered_df["fda_year"].isna() & filtered_df['at_least_one_phase4']].shape

(753, 17)

In [45]:
filtered_df["fda_AP"].value_counts()


fda_AP
False    3478
True     1309
Name: count, dtype: int64

In [46]:
filtered_df[
    (filtered_df["clinical_count"] > 1) &
    (filtered_df["preclinical_count"] == 1)
]


,normalized_key,clinical_doc_ids,phase,overall_status,clinical_count,preclinical_doc_ids,disease,preclinical_count,max_phase,at_least_one_phase3,at_least_one_phase3_completed,at_least_one_phase4,fda_merged_umls_label,fda_year,fda_sponsor_name,fda_application_number,fda_AP
151,avonex,"[NCT00030966, NCT00037102, NCT00037115, NCT000...","[Phase 3, Phase 4, Phase 4, Phase 2, Phase 4, ...","[Completed, Completed, Withdrawn, Completed, W...",37.0,[15081262],[multiple sclerosis],1.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
207,ipv,"[NCT00161928, NCT00303316, NCT00319852, NCT003...","[Phase 3, Phase 3, Phase 3, Phase 3, Phase 3, ...","[Completed, Completed, Completed, Completed, C...",28.0,[35651037],[type 2 diabetes mellitus],1.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
222,chantix,"[NCT00502216, NCT00525837, NCT00567008, NCT005...","[Phase 1/Phase 2, Not Applicable, Phase 2, Pha...","[Completed, Completed, Completed, Completed, C...",26.0,[29467844],[pediatric osteosarcoma],1.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
251,flortaucipir f-18,"[NCT01992380, NCT02051764, NCT02167594, NCT022...","[Phase 1, Phase 2, Phase 1, Phase 1, Phase 2, ...","[Completed, Terminated, Completed, Completed, ...",24.0,[35596745],[alzheimer disease],1.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
279,abilify,"[NCT00174876, NCT00181779, NCT00194012, NCT001...","[Phase 2, Phase 4, Phase 3, Phase 4, Phase 3, ...","[Completed, Completed, Completed, Completed, C...",21.0,[25323073],[schizophrenia],1.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2570,tv-46000,"[NCT03503318, NCT03893825]","[Phase 3, Phase 3]","[Completed, Completed]",2.0,[35245409],[schizophrenia],1.0,Phase 3,True,True,False,NaN,NaN,NaN,NaN,False
2571,vx15/2503,"[NCT01764737, NCT02481674]","[Phase 1, Phase 2]","[Completed, Completed]",2.0,[25657333],[multiple],1.0,Phase 2,False,False,False,NaN,NaN,NaN,NaN,False
2572,xarelto,"[NCT02042534, NCT02970773]","[Phase 2, Phase 4]","[Completed, Withdrawn]",2.0,[35675485],[type 2 diabetes mellitus],1.0,Phase 4,False,False,True,NaN,NaN,NaN,NaN,False
2573,zilucoplan,"[NCT04297683, NCT04436497]","[Phase 2/Phase 3, Phase 2/Phase 3]","[Recruiting, Completed]",2.0,[36009583],[immune-mediated necrotizing myopathy],1.0,Phase 2/Phase 3,True,True,False,NaN,NaN,NaN,NaN,False


In [47]:
filtered_df[filtered_df["fda_year"].isna() & filtered_df['at_least_one_phase4']].head(10)

,normalized_key,clinical_doc_ids,phase,overall_status,clinical_count,preclinical_doc_ids,disease,preclinical_count,max_phase,at_least_one_phase3,at_least_one_phase3_completed,at_least_one_phase4,fda_merged_umls_label,fda_year,fda_sponsor_name,fda_application_number,fda_AP
0,placebo,"[NCT00006289, NCT00030147, NCT00035529, NCT000...","[Phase 2, Phase 4, Phase 2, Phase 2, Phase 4, ...","[Terminated, Completed, Terminated, Completed,...",330.0,"[33767693, 33767693, 33767693, 33767693]","[severe sepsis, septic shock, systemic inflamm...",4.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
13,antidepressant,"[NCT00009191, NCT00015548, NCT00018902, NCT000...","[Phase 4, Not Applicable, Phase 2/Phase 3, Pha...","[Completed, Completed, Completed, Completed, C...",151.0,"[29157077, 22464799, 22464799, 15154022, 15154...","[depressive disorder, stroke disorder, poststr...",267.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
22,melatonin,"[NCT00097474, NCT00113906, NCT00230737, NCT002...","[Phase 2, Phase 2/Phase 3, Phase 2, Phase 2, N...","[Completed, Completed, Completed, Completed, U...",126.0,"[31738882, 31738882, 31738882, 11860741, 31187...","[allergic respiratory disease, allergic asthma...",7890.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
27,lithium,"[NCT00010868, NCT00015769, NCT00025792, NCT000...","[Phase 2, Phase 2, Phase 2, Phase 2, Phase 2, ...","[Completed, Completed, Completed, Completed, C...",119.0,"[15691529, 31193305, 11907184, 11907184, 31867...","[hypoxia, bipolar disorder, alcohol-induced di...",2847.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
28,"fatty acids, omega-3","[NCT00010868, NCT00096798, NCT00114595, NCT001...","[Phase 2, Phase 3, Phase 4, Phase 3, Not Appli...","[Completed, Completed, Completed, Completed, C...",119.0,"[15916761, 26374481, 26374481, 26374481, 26374...","[age-related macular degeneration, ischemic di...",1702.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
31,steroid,"[NCT00033189, NCT00203047, NCT00203294, NCT002...","[Phase 2, Phase 4, Not Applicable, Phase 2/Pha...","[Completed, Terminated, Completed, Terminated,...",114.0,"[31740635, 31740635, 31740635, 31740635, 31921...","[dysfunction, meibomitis, meibomian gland dysf...",2032.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
33,antipsychotic,"[NCT00014001, NCT00018668, NCT00044655, NCT000...","[Phase 4, Phase 4, Phase 4, Phase 2, Phase 4, ...","[Completed, Completed, Completed, Completed, C...",107.0,"[19059234, 17891479, 28472198, 28472198, 28472...","[schizophrenia, neuralgia, major depressive di...",264.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
40,botulinum toxin,"[NCT00133861, NCT00173745, NCT00175123, NCT002...","[Phase 2/Phase 3, Not Applicable, Phase 4, Not...","[Completed, Unknown status, Unknown status, Te...",98.0,"[26980040, 26980040, 26980040, 11975099, 11975...",[permanent neonatal diabetes mellitus-pancreat...,227.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
43,adrenal cortex hormone,"[NCT00122278, NCT00283309, NCT00284583, NCT002...","[Phase 3, Phase 4, Not Applicable, Phase 3, Ph...","[Completed, Terminated, Unknown status, Comple...",94.0,"[9451464, 8210334, 8210334, 26646781, 26646781...","[listeriosis, radiation pneumonitis, interstit...",2854.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False
58,antidepressants,"[NCT00009191, NCT00009672, NCT00062738, NCT000...","[Phase 4, Phase 2, Phase 2, Phase 3, Phase 2, ...","[Completed, Completed, Completed, Completed, C...",78.0,"[3756328, 22464799, 22464799, 24310907, 588211...","[epilepsy, stroke disorder, poststroke depress...",457.0,Phase 4,True,True,True,NaN,NaN,NaN,NaN,False


In [48]:
filtered_df.normalized_key.nunique()

4785

## Report translational stats

In [49]:
from typing import Optional, List


In [50]:
def _is_linked(e, mapping_lookup):
    if e not in mapping_lookup:
        print(f"{e}: not_found")
    mid = mapping_lookup.get(e, "-1")
    #if mid == "-1":
     #   print(f"not linked entity: {e}")
    return str(mid).strip() != "-1"

def _is_short(e, mapping_lookup):
    if len(e.strip()) < 3:
        print(f"{e}, {_is_linked(e,mapping_lookup)}")

    return len(e.strip()) < 3

In [59]:
def clinical_reach_report(
    preclinical_df: pd.DataFrame,
    filtered_df: pd.DataFrame,
    *,
    drugs_col_to_use: str,
    mapping_lookup: dict,
    fda_preclin_not_clin: Optional[List[str]] = None,
    key_col: str = "normalized_key",
    phase4_col: str = "at_least_one_phase4",
    phase3_completed_col: str = "at_least_one_phase3_completed",
    fda_approval_col: str = "fda_AP",
    ignore_id: str = "-1",
    case_insensitive: bool = False,
):
    # optionally lowercase mapping keys once
    if case_insensitive:
        mapping_lookup = {str(k).lower(): v for k, v in mapping_lookup.items()}

    def fmt(n): return f"{int(n):,}"
    def pct(a, b): return (a / b * 100) if b else 0.0

     # --- denominators ---
    preclin_unique = preclinical_df[drugs_col_to_use].dropna().unique()
    total_unique_preclin = len(preclin_unique)
    
    clinical_unique = set(filtered_df[key_col].dropna().astype(str))
    
    # preclinical counts
    preclin_series = preclinical_df[drugs_col_to_use].dropna().astype(str)
    preclin_counts = preclin_series.value_counts()
    
    preclin_gt1 = set(preclin_counts[preclin_counts > 1].index)
    preclin_eq1 = set(preclin_counts[preclin_counts == 1].index)
    
    # rule:
    # (>1 preclinical) OR (==1 preclinical AND appears in clinical at least once)
    total_unique_preclin_gt1 = len(
        preclin_gt1 | (preclin_eq1 & clinical_unique)
    )
    
    total_preclinical_linked = sum(
        _is_linked(e, mapping_lookup)
        for e in preclin_unique
    )
    
    # set of drugs that satisfy the gt1 rule
    preclin_gt1_rule = preclin_gt1 | (preclin_eq1 & clinical_unique)
    
    # linked subset of that
    total_preclinical_linked_gt1 = sum(
        _is_linked(e, mapping_lookup)
        for e in preclin_gt1_rule
    )

    
    # --- reached clinical ---
    # drugs either in CT.gov dataset, or with a link to FDA
    reached_clinical_entities = clinical_unique | fda_preclin_not_clin
    reached_clinical = len(clinical_unique) + len(fda_preclin_not_clin)

    reached_clinical_and_linked = sum(
        _is_linked(e, mapping_lookup)
        for e in clinical_unique
    )
    
    # --- phase 3 completed / phase 4 / FDA ---
    df_p3c_or_p4_or_fda = filtered_df[
        filtered_df[phase3_completed_col].fillna(False)
        | filtered_df[phase4_col].fillna(False)
        | filtered_df[fda_approval_col].fillna(False)
    ]

    p3c_or_p4_unique = df_p3c_or_p4_or_fda[key_col].dropna().unique()

    reached_p3c_or_p4_or_fda = len(p3c_or_p4_unique) + len(fda_preclin_not_clin)
    reached_p3c_or_p4_and_linked = sum(
        _is_linked(e, mapping_lookup)
        for e in p3c_or_p4_unique
    )

    # --- print report ---
    print("\n" + "=" * 70)
    print("Clinical reach report (preclinical → clinical) [flat entities]")
    print("=" * 70)

    print("\nDenominators (preclinical diversity):")
    print(f"Note excl. infrequent rule: (>1 preclinical) OR \n (==1 preclinical AND >=1 clinical)")
    print(f"  Unique drugs:                                  {fmt(total_unique_preclin)}")
    print(f"  Unique drugs excl. infrequent:                 {fmt(total_unique_preclin_gt1)}")
    print(f"  Unique drugs successfully linked:               {fmt(total_preclinical_linked)}")
    print(f"  Unique drugs linked excl. infrequent:           {fmt(total_preclinical_linked_gt1)}")
    print(f"  Unique drugs from FDA:                          {fmt(len(fda_preclin_not_clin))}")

    print("\nReach metrics (unique drugs):")
    print(f"  Reached clinical (any / linked):        {fmt(reached_clinical)} / {fmt(reached_clinical_and_linked)}")
    print(f"    = {pct(reached_clinical, total_unique_preclin):.2f}% of all unique")
    print(f"    = {pct(reached_clinical, total_unique_preclin_gt1):.2f}% excl. entities only once in animal DS")
    print(f"    = {pct(reached_clinical_and_linked, total_preclinical_linked):.2f}% of linked unique")
    print(f"    = {pct(reached_clinical_and_linked, total_preclinical_linked_gt1):.2f}% of linked unique excl. entities only once in animal DS")

    print(f"\n  Reached (P3 completed / P4 / FDA) (any / linked):     {fmt(reached_p3c_or_p4_or_fda)} / {fmt(reached_p3c_or_p4_and_linked)}")
    print(f"    = {pct(reached_p3c_or_p4_or_fda, total_unique_preclin):.2f}% of all unique")
    print(f"    = {pct(reached_p3c_or_p4_or_fda, total_unique_preclin_gt1):.2f}% excl. entities only once in animal DS")
    print(f"    = {pct(reached_p3c_or_p4_and_linked, total_preclinical_linked):.2f}% of linked unique")
    print(f"    = {pct(reached_p3c_or_p4_and_linked, total_preclinical_linked_gt1):.2f}% of linked unique excl. entities only once in animal DS")

    print("\n" + "-" * 70)
    print("Note: percentages are over unique entity types (not mention frequency).")
    print("-" * 70)

    return {
        "total_unique_preclin": total_unique_preclin,
        "total_unique_preclin_gt1": total_unique_preclin_gt1,
        "total_preclinical_linked": total_preclinical_linked,
        "reached_clinical": reached_clinical,
        "reached_clinical_and_linked": reached_clinical_and_linked,
        "reached_phase3_completed_or_phase4_or_fda": reached_p3c_or_p4_or_fda,
        "reached_phase3_completed_or_phase4_and_linked": reached_p3c_or_p4_and_linked,
    }, reached_clinical_entities

In [60]:
preclinical_df.head()

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
0,31733831,asthma,asthma,MONDO:0004979,asthma,-1,asthma,MONDO:0004979,isorhynchophylline,isorhynchophylline,C0245133,-1,isorhynchophylline,C0245133
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,"mirn1192 microrna, mouse",C3849422
2,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,antgomir-1192,-1
3,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,agomir-1192,-1
4,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,-1,systemic lupus erythematosus,MONDO:0007915,hla-g2|g2,HLA-G2 Isoform|g2,C0967254|-1,-1,hla-g2 isoform,C0967254


In [66]:
clinical_df[
    clinical_df['merged_umls_label']
    .str.contains("momelotinib", case=False, na=False)
]

,nct_id,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,disease<>drug,phase,overall_status


In [61]:
drugs_col_to_use

'merged_umls_label'

In [62]:
report, reached_clinical_entities = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use=drugs_col_to_use,
    mapping_lookup=mapping_lookup,
    fda_preclin_not_clin=fda_preclin_not_clin
)



Clinical reach report (preclinical → clinical) [flat entities]

Denominators (preclinical diversity):
Note excl. infrequent rule: (>1 preclinical) OR 
 (==1 preclinical AND >=1 clinical)
  Unique drugs:                                  291,457
  Unique drugs excl. infrequent:                 182,017
  Unique drugs successfully linked:               57,160
  Unique drugs linked excl. infrequent:           44,106
  Unique drugs from FDA:                          1,436

Reach metrics (unique drugs):
  Reached clinical (any / linked):        6,221 / 3,649
    = 2.13% of all unique
    = 3.42% excl. entities only once in animal DS
    = 6.38% of linked unique
    = 8.27% of linked unique excl. entities only once in animal DS

  Reached (P3 completed / P4 / FDA) (any / linked):     3,945 / 2,070
    = 1.35% of all unique
    = 2.17% excl. entities only once in animal DS
    = 3.62% of linked unique
    = 4.69% of linked unique excl. entities only once in animal DS

-------------------------

In [55]:
n_in_clinical_linked = sum(_is_linked(e, mapping_lookup) for e in list(filtered_df.normalized_key))
n_in_clinical_linked

3651

In [64]:
len(reached_clinical_entities)

6221

In [65]:
import json

with open("out/reached_clinical_drug_entities.json", "w") as f:
    json.dump(list(reached_clinical_entities), f)
